# Demo ollama local



### Prérequis : étapes à réaliser dans le terminal

Avant de lancer ce notebook, les étapes suivantes doivent avoir été effectuées **dans un terminal** sur votre machine :

#### Installer Ollama
Télécharger et installer Ollama depuis [https://ollama.com/download](https://ollama.com/download).

#### Télécharger un modèle

Télécharger le modèle `llama3.2:1b` (~1,3 Go) (À n'effectuer qu'une seule fois) :

```bash
ollama pull llama3.2:1b
```

(D'autres modèles sont disponibles sur [https://ollama.com/library](https://ollama.com/library))

#### Démarrer le serveur Ollama

```bash
ollama serve
```

Le serveur écoute par défaut sur `http://127.0.0.1:11434`. **Laisser ce terminal ouvert** pendant toute la session.

#### Installer le package Python
Dans votre notebook ou environnement :
```bash
pip install ollama
```

---

Une fois ces étapes réalisées, vous pouvez exécuter les cellules ci-dessous.

In [ ]:
# Création de notre jeu de données (25 phrases)

import pandas as pd

test_data = [
    {
        "text": "Cette réforme est une excellente opportunité pour notre économie.",
        "label": "POSITIF",
    },
    {
        "text": "Nous soutenons pleinement cette initiative très prometteuse.",
        "label": "POSITIF",
    },
    {
        "text": "Le projet de loi est un désastre total et une honte.",
        "label": "NEGATIF",
    },
    {"text": "Je suis extrêmement déçu par ces mesures injustes.", "label": "NEGATIF"},
    {
        "text": "Le ministre a annoncé la date des prochaines élections.",
        "label": "NEUTRE",
    },
    {
        "text": "Quelle brillante idée d'augmenter les taxes de 20%, un pur coup de génie !",
        "label": "NEGATIF",
    },
    {
        "text": "Bravo au gouvernement pour cette inflation insupportable, on adore !",
        "label": "NEGATIF",
    },
    {
        "text": "Un triomphe absolu que d'avoir réussi à faire grève pendant 3 mois...",
        "label": "NEGATIF",
    },
    {
        "text": "Une manifestation violente a rassemblé 10 000 personnes en colère contre le gouvernement.",
        "label": "NEUTRE",
    },
    {
        "text": "Le scandale de corruption a éclaté ce matin à la Une des journaux.",
        "label": "NEUTRE",
    },
    {
        "text": "La crise économique provoque la fermeture de nombreuses entreprises dans la région.",
        "label": "NEUTRE",
    },
    {
        "text": "L'accord historique a été signé dans un climat de tension extrême.",
        "label": "NEUTRE",
    },
    {
        "text": "Le député a attaqué violemment le bilan de son adversaire lors du débat.",
        "label": "NEUTRE",
    },
    {
        "text": "Il serait très exagéré de dire que cette mesure est mauvaise.",
        "label": "POSITIF",
    },
    {
        "text": "Je ne peux pas m'empêcher de penser que ce n'est pas complètement un échec.",
        "label": "POSITIF",
    },
    {"text": "Pas exactement le succès du siècle.", "label": "NEGATIF"},
    {
        "text": "Rien ne prouve que ce texte soit une erreur historique.",
        "label": "POSITIF",
    },
    {"text": "On a déjà vu pire comme proposition de loi.", "label": "POSITIF"},
    {
        "text": "La loi autorise la fermeture des commerces le dimanche.",
        "label": "NEUTRE",
    },
    {
        "text": "Le président a annulé sa visite prévue dans les territoires sinistrés.",
        "label": "NEUTRE",
    },
    {
        "text": "Les impôts augmenteront de 2% pour financer la sécurité sociale.",
        "label": "NEUTRE",
    },
    {
        "text": "Le texte final est inapplicable bien qu'il y ait quelques bonnes idées.",
        "label": "NEGATIF",
    },
    {
        "text": "Un débat long et ennuyeux, mais qui accouche d'une mesure très positive.",
        "label": "POSITIF",
    },
    {
        "text": "Malgré les protestations, l'assemblée a validé un texte fondamental pour nos libertés.",
        "label": "POSITIF",
    },
    {
        "text": "C'est un véritable triomphe historique pour l'opposition qui a su imposer ses idées.",
        "label": "POSITIF",
    },
]

test_df = pd.DataFrame(test_data)
test_df.head()


## Connecter avec notre ollama local

In [ ]:
# %pip install ollama

In [ ]:
# (à lancer après avoir lancé le serveur ollama avec la commande `ollama serve` dans votre terminal)
from ollama import Client

client = Client(host="http://127.0.0.1:11434")

In [ ]:
# mini test
# passer un modèle que l'on a déjà téléchargé (ex: `ollama pull llama3.2:1b`)
response = client.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "user",
            "content": "Why is the sky blue?",
        },
    ],
)

In [ ]:
print(response)

In [ ]:
# fonction de génération du prompt pour la tache de classification
def get_prompt(text):
    content_system = """
Tu es un expert de sciences politiques travaillant sur des déclarations politiques.

Ton travail est de classifier chaque texte comme :
- "POSITIF"
- "NEUTRE"
- "NEGATIF"

Réponds uniquement avec un de ces trois labels.
Ne produis aucun autre texte.
Ne produis aucune ponctuation.
Réponds par un seul mot.
"""

    return [
        {"role": "system", "content": content_system},
        {"role": "user", "content": f'Classifie cette déclaration :\n"{text}"'},
    ]


# fonction de prédiction du label
def predict_label(text, model="llama3.2:1b"):
    response = client.chat(model=model, messages=get_prompt(text))

    return response["message"]["content"].strip()

In [ ]:
# lancer la prédiction sur le jeu de test (appliquer la fonction avec `apply`))
test_df["prediction"] = test_df["text"].apply(
    lambda x: predict_label(x, model="llama3.2:1b")
)

In [ ]:
# afficher le tout et voir si on est content
test_df